# D8客群与报告

In [ ]:
# ── 第一格：环境准备 ──
%load_ext autoreload
%autoreload 2
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, db, geo, pipeline, viz
from signal_insight.viz import (MAIN, MAIN_LIGHT, MAIN_DARK, 
            MAIN_MID, GRAY, GRAY_DARK, INK, MUTED, FAINT, SEG_COLOR)

viz.apply_theme()
con = sqlite3.connect(config.DB_PATH)
DAY = config.ANALYSIS_DAY
DAY_START = pd.Timestamp(DAY)
NB = "D08_客群与报告"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"
REPORT = config.PROJECT_ROOT / "report" # 报告和材料包放这里；只放聚合结果，可以进 Git
REPORT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 30)

pipeline.run_day()      # 上游表刷到最新（幂等），和 D07 第二格一样
P = config.load_settings()
hot = pd.read_sql("SELECT * FROM dim_hotspot", con).iloc[0]
R = P["hotspot"]["radius_m"]


## 是谁：用规则给访客分群
需求：傍晚的访客的是一群什么人？按过夜地给他分为四类
- 过夜地不明：识别不出过夜地
- 市外来访：过夜地在沈阳市城外
- 周边居民：过夜地距离西塔不到1.5公里
- 市内来访：其余

In [ ]:
stay = pd.read_sql(
    "SELECT * FROM dws_stay",con,
            parse_dates=["start_time", "end_time"])
vis = pd.read_sql(
    "SELECT * FROM dws_visitor",con).set_index("imsi")  # D07 写的：每个傍晚访客一行
night = pd.read_sql(
    "SELECT * FROM dws_night",con).set_index("imsi")  # D07 写的：过夜地

# 让数据库去重计数，不用把 51 万行读进来
n_users = pd.read_sql(
    "SELECT COUNT(DISTINCT imsi) AS n FROM dwd_signal",con)["n"][0]  
print(
    f"用户 {n_users:,}；停留 {len(stay):,}；傍晚访客 {len(vis):,}；有过夜地的用户 {len(night):,}"
)


- 怎么分的问题：
    - 先判不明：d_km是NaN的话，后面的<1.5比较结果是False,
    - 再判断市外：
    - 再判是不是周边：<1.5
    - 都不是：市内来访

In [ ]:
# 过夜第离西塔的距离换算成公里
d_km = vis['night_dist_xita_m'] / 1000

# np.select(条件列表, 取值列表, default=其余)：对每一行，从前往后看条件，取第一个成立的条件对应的值。
# 所以四行的顺序本身就是业务规则：
#   ① 先判“不明”：d_km 是 NaN 的话，后面的“< 1.5”比较结果是 False，不放第一位也不会出错，
#      但放第一位表达的是态度——先承认“不知道”，再谈分类
#   ② 再判“市外”：过夜地在市外的人，哪怕碰巧离西塔近，也不算周边居民（顺序换了，结果就变，讲师会演示）
#   ③ 再判“周边”：< 1.5 公里
#   ④ 都不是：市内来访
vis['segment'] = np.select(
    [d_km.isna(), vis['night_district'] == '市外',d_km < P['visitor']['near_km']],
    ['过夜地不明','市外来访','周边居民'],
    default="市内来访"
)

seg =  vis.groupby("segment").agg(
    人数=('segment','size'),
    西塔停留中位_分 = ('xita_stay_min', "median"),
    首次到达中位_时 = ("arrive_hour", 'median')
)
seg['占比'] = seg['人数'] / seg['人数'].sum()
seg.sort_values('人数', ascending = False)

In [ ]:
# ── 出图 D8_01：横向条形图（四类比长短，不用饼图；颜色用 viz.SEG_COLOR 的固定映射，和后面所有图一致）──
s_ = vis["segment"].value_counts()
order = s_.sort_values().index
fig, ax = plt.subplots(figsize=(10, 5.0), layout="constrained")
bars = ax.barh(order, s_[order], color=[SEG_COLOR[k] for k in order], height=0.6)
ax.bar_label(bars, labels=[f"{s_[k] / len(vis):.0%}（{s_[k]:,} 人）" for k in order], padding=4, fontsize=10, color=INK)
ax.set_xlim(0, s_.max() * 1.35); ax.set_xticks([]); ax.grid(False); ax.tick_params(axis="y", labelsize=10.5, labelcolor=INK)
ax.text(s_.max() * 0.3, list(order).index("市外来访"), "← 国庆假期的商圈，外地人只有 1%？下一节换个口径再看",
        fontsize=9.5, color=MUTED, va="center")
top_ = s_.idxmax()
title = f"傍晚访客里 {s_[top_] / len(vis):.0%} 是{top_}，{s_['周边居民'] / len(vis):.0%} 是周边居民"
viz.finish(fig, title,
           f"{len(vis):,} 个傍晚访客按过夜地分类：周边居民 = 过夜地离西塔 1.5 公里内；灰色 = 识别不出过夜地，不是一类人；口径：dws_visitor",
           SOURCE, "D8_01_傍晚访客客群.png")
plt.show()


In [ ]:
# ── 西塔客群画像：谁来、待多久、几点到（接在 viz.apply_theme() 之后运行）──
import hashlib

NB = "D07_客群画像"            # 本 notebook 名：进来源行和文件名，按实际改
UNKNOWN = "过夜地不明"          # 不是一类人，是“不知道”：单独放最后、永远灰色

# ---------- 1. 规范第 6 节的数字格式 ----------
def pct(x):
    """一般取整；不到 1% 多留一位，不到 0.1% 留两位（0.01%，不写 0.0%）。"""
    return f"{x:.0%}" if x >= 0.01 else f"{x:.{1 if x >= 0.001 else 2}%}"

def dur(minutes):
    """超过 2 小时改用小时。"""
    return f"{minutes / 60:.1f} 小时" if minutes >= 120 else f"{minutes:.0f} 分钟"

def hhmm(h):
    m = int(round(h * 60))
    return f"{m // 60:02d}:{m % 60:02d}"

def period(h):
    return next(name for end, name in [(5, "凌晨"), (11, "上午"), (13, "中午"),
                                       (17, "下午"), (19, "傍晚"), (24.01, "晚上")] if h < end)


# ---------- 2. 标题、副标题、标注里的数全部由代码算 ----------
known = seg.drop(index=UNKNOWN)
order = known["人数"].sort_values(ascending=False).index.tolist() + [UNKNOWN]
top = order[0]                                         # 能识别过夜地的人里最多的一类
N, n_known = int(seg["人数"].sum()), int(known["人数"].sum())
small = known["人数"].idxmin()
arrive, stay = seg.at[top, "首次到达中位_时"], seg.at[top, "西塔停留中位_分"]

title = f"{top}是西塔主客群：{period(arrive)} {hhmm(arrive)} 前后到，停留至少 {dur(stay)}"
subtitle = (f"{config.ANALYSIS_DAY[5:]} 在西塔有停留的 {N:,} 人，按过夜地分 {len(seg)} 类；"
            f"停留时长、首次到达时刻取各类中位数\n"
            f"能识别过夜地的占 {pct(n_known / N)}；停留只算当天信令，跨 0 点的部分被截断，故写“至少”；"
            f"{small}仅 {int(seg.at[small, '人数'])} 人，中位数仅供参考")
cfg = config.load_settings()
cfg_tag = hashlib.sha256(config.SETTINGS_PATH.read_bytes()).hexdigest()[:8]
cfg_tag = f"{cfg['version']}+{cfg_tag}" if cfg.get("version") else cfg_tag
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"


# ---------- 3. 画图：三栏共用一套行，读一行就是一类人的完整画像 ----------
y = np.r_[np.arange(len(order) - 1), len(order) - 0.6]  # “不明”和上面隔开一点
colors = [SEG_COLOR[k] for k in order]
txt_color = [MUTED if k == UNKNOWN else INK for k in order]
weight = ["bold" if k == top else "normal" for k in order]
divider = y[-1] - 0.7

fig, axes = plt.subplots(1, 3, figsize=(10, 4.6), sharey=True, layout="constrained",
                         gridspec_kw={"width_ratios": [1.25, 1.1, 1.15]})
fig.get_layout_engine().set(w_pad=0.15)

# ① 人数与占比：条形图从 0 开始，数字直接写在条尾
ax = axes[0]
n = seg.loc[order, "人数"]
ax.barh(y, n, color=colors, height=0.62)
for yi, k, c, w in zip(y, order, txt_color, weight):
    ax.text(n[k] + n.max() * 0.03, yi, f"{n[k]:,} 人 · {pct(seg.at[k, '占比'])}",
            va="center", fontsize=10, color=c, fontweight=w)
ax.set_xlim(0, n.max() * 1.55)
ax.set_title(f"人数 · 占全部 {N:,} 人", loc="left", fontsize=10.5, color=INK, pad=8)

# ② 停留时长：中位数，删失写“至少”
ax = axes[1]
h = seg.loc[order, "西塔停留中位_分"] / 60
ax.barh(y, h, color=colors, height=0.62)
for yi, k, c, w in zip(y, order, txt_color, weight):
    ax.text(h[k] + h.max() * 0.03, yi, f"至少 {dur(h[k] * 60)}",
            va="center", fontsize=10, color=c, fontweight=w)
ax.set_xlim(0, h.max() * 1.6)
ax.set_title("西塔停留时长 · 中位数", loc="left", fontsize=10.5, color=INK, pad=8)

# ③ 首次到达时刻：放在 0–24 时的时间轴上
ax = axes[2]
t = seg.loc[order, "首次到达中位_时"]
ax.hlines(y, 0, 24, color=GRAY, lw=0.8, alpha=0.6, zorder=1)
ax.scatter(t, y, s=70, color=colors, edgecolor="white", linewidth=1.2, zorder=3)
for yi, k, c, w in zip(y, order, txt_color, weight):
    note = "（夜里已在）" if t[k] < 5 else ""
    ax.annotate(f"{hhmm(t[k])}{note}", (t[k], yi), xytext=(0, 9), textcoords="offset points",
                ha="left" if t[k] < 5 else "center", va="bottom", fontsize=10, color=c, fontweight=w)
ax.set_xlim(-0.5, 24.5)
ax.set_xticks(range(0, 25, 6), [f"{v} 时" for v in range(0, 25, 6)])
ax.grid(axis="x", lw=0.6)
ax.set_title("当天首次到达时刻 · 中位数", loc="left", fontsize=10.5, color=INK, pad=8)

# 公共：行名只在最左栏，“不明”行灰字；能识别 / 不能识别之间一条虚线
for ax in axes:
    ax.grid(axis="y", visible=False)
    ax.axhline(divider, color=GRAY, lw=0.8, ls=(0, (3, 3)))
    ax.tick_params(axis="y", length=0, labelsize=10.5)
for ax in axes[:2]:
    ax.xaxis.set_visible(False)                        # 数值已直接标在条尾，不再要刻度
    ax.spines["bottom"].set_visible(False)
axes[0].set_yticks(y, order)
axes[0].set_ylim(y[-1] + 0.55, -0.75)                  # 倒序：人数最多的在最上
for lab, c in zip(axes[0].get_yticklabels(), txt_color):
    lab.set_color(c)

out = viz.finish(fig, title, subtitle, SOURCE, f"{NB[:3]}_01_西塔客群画像.png")
plt.show()

## 数字反常那就查口径：外地来客只有1%
需求：国庆假期都核心商圈，外地人只占1%，不合常理，第一反应差口径，不是改数字，也不是照抄进报告

追查：判断外地人只看过夜地。当天白天才从外地进城的人，凌晨还在外地，信令根本不在样本范围内，于是识别不出过夜地，被归进了“过夜地不明”。
修正：加一条独立证据：当天第一条信令在市域外、之后进入市域，就是外地来客。

In [ ]:
sig_d = pd.read_sql("""
SELECT s.imsi, s.ts, c.district FROM dwd_signal s JOIN dim_cell c ON s.lac_cell = c.lac_cell
""", con, parse_dates=['ts'])
sig_d = sig_d.sort_values(['imsi','ts'], kind='stable')

first = sig_d.groupby('imsi').first()
# 先筛出： 在市域内的信令，再取每人第一条 = 第一次出现在市域的哪一条
entry = sig_d[sig_d['district'] == '市外'].groupby('imsi').first()

In [ ]:
# 入城来客= 第一条在市外 切 之后进过 市域
inbound = first.index[first['district'] == '市外'].intersection(entry.index)

vis['inbound'] = vis.index.isin(inbound)
vis['segment_v2'] = np.where(
    vis['inbound'] | (vis['segment'] == '市外来访'), '外地来客', vis['segment'])
ORDER = ['周边居民', '市内来访', '外地来客','过夜地不明']
old = vis['segment'].replace(
    {"市外来访":'外地来客'}).value_counts().reindex(ORDER, fill_value=0)
new = vis['segment_v2'].value_counts().reindex(ORDER, fill_value=0)
moved = int((vis['inbound'] & (vis['segment'] == '过夜地不明')).sum())
print(f"当天从市外进城 {len(inbound):,} 人，其中 {vis['inbound'].sum():,} 人是傍晚访客；原来被归进“过夜地不明”的 {moved} 人")
pd.DataFrame({"原口径（按过夜地）": old, "修正后（加入城轨迹）": new})

In [ ]:
# ── 出图 D8_02：同一批人、两种口径，每类上下两根条形对比 ──
fig, ax = plt.subplots(figsize=(10, 5.0), layout="constrained")
y = np.arange(len(ORDER))
ax.barh(y - 0.19, new.values, height=0.36, color=[SEG_COLOR[k] for k in ORDER], label="加入城轨迹（修正后）")
ax.barh(y + 0.19, old.values, height=0.36, color=GRAY, label="按过夜地（8.1）")
for yy, o_, n_ in zip(y, old.values, new.values):
    ax.text(n_ + 15, yy - 0.19, f"{n_:,}（{n_ / len(vis):.0%}）", va="center", fontsize=10, color=INK, fontweight="bold")
    ax.text(o_ + 15, yy + 0.19, f"{o_:,}", va="center", fontsize=9, color=MUTED)
ax.set_yticks(y, ORDER, fontsize=10.5); ax.invert_yaxis()
ax.set_xlim(0, max(old.max(), new.max()) * 1.3); ax.set_xticks([]); ax.grid(False)
ax.legend()
title = f"按过夜地只找到 {old['外地来客']} 个外地来客，看入城轨迹是 {new['外地来客']} 人（{new['外地来客'] / len(vis):.0%}）"
viz.finish(fig, title,
           f"{len(vis):,} 个傍晚访客，每类上彩下灰：彩 = 加上当天入城轨迹，灰 = 只按过夜地；其中 {moved} 人从“过夜地不明”改判为外地来客；\n"
           "外地来客 = 当天第一条信令在沈阳市域外、之后进入市域（或过夜地在市外）；口径：dws_visitor、dwd_signal × dim_cell",
           SOURCE, "D8_02_外地来客口径修正.png")
plt.show()

仅按过夜地判断，傍晚访客中外地来客仅 31 人（1%）；加入当天入城轨迹后为 473 人（14%），其中 436 人原被归为“过夜地不明”（图 D8_02）。

In [ ]:
db.replace_table(con, 
    vis[["segment", "inbound", "segment_v2"]].reset_index(), "dws_visitor_tag")

# 修正后的客群画像，写进 ads_ 层（对外表）
seg2 = vis.groupby("segment_v2").agg(
    人数=("segment_v2", "size"),
    西塔停留中位_分=("xita_stay_min", "median"),
    首次到达中位_时=("arrive_hour", "median")).reindex(ORDER)
seg2["占比"] = (seg2["人数"] / seg2["人数"].sum()).round(3)
seg2.rename_axis("客群").reset_index().to_sql(
    "ads_visitor_segment", con, if_exists="replace", index=False)

In [ ]:
seg2

## 还去了哪？来西塔前，离开之后
需求：傍晚访客来西塔之前在哪儿待过？离开之后又去了哪？甲方关心三个商圈之间有没有联动。

做法：每个傍晚访客，找到傍晚那次西塔停留的开始时刻（记为“到达”）。“之前” = 在到达前结束的、西塔以外的停留；“之后” = 到达之后开始的停留。每个停留归一类：过夜地（离过夜地 500 米内）、太原街商圈、中街商圈、其他地点。一个人可以属于几类，所以占比加起来超过 100%。

In [ ]:
# 1. 只要傍晚访客的停留： join按照imsi把过夜地坐标贴到每个停留上面，没有过夜地的就是Nan
stay = pd.read_sql(
    "SELECT * FROM dws_stay",con,
            parse_dates=["start_time", "end_time"])
ev = stay[stay['imsi'].isin(vis.index)].join(
    night[['night_lng', 'night_lat']],on='imsi')

# 每个停留离这个人的过夜地多远；≤ 500 米算“在过夜地”。没有过夜地的人自然不算
ev['home'] = geo.haversine(ev['lat'], ev['lng'], ev['night_lat'],ev['night_lng']) <= R

# 2. 每人的到达时刻：傍晚那次西塔停留：距离西塔500米内，结束大于17:00，里最早的开始时刻

arrive = (ev[(ev['dist_xita_m'] <= R) & 
    (ev['end_time'] >= DAY_START +
    pd.Timedelta(hours=P['visitor']['evening_hour']))]
        .groupby('imsi')['start_time'].min().rename('arrive')
         )
ev = ev.join(arrive, on='imsi')
AOI2 = ['太原街商圈', "中街商圈"]

In [ ]:
def place_flags(s):
    """把一批停留按地点归类，再按人会汇总成去没去过"""
    in_aoi = s['aoi'].isin(AOI2)
    f = pd.DataFrame({
        "imsi":s['imsi'],
        "过夜地": s['home'] & ~in_aoi,
        '太原街商圈': s['aoi'] == '太原街商圈',
        '中街商圈': s['aoi'] == '中街商圈'
    })
    f['其他地点'] = ~f.drop(columns='imsi').any(axis=1)
    return f.groupby('imsi').any()

# 3. 之前：在到达之前结束，且不在西塔圈内，之后：在到达之后开始（商圈和西塔圈有少量重叠，商圈停留照算）
before_s = ev[(ev['end_time'] <= ev['arrive']) &
              (ev['dist_xita_m'] > R)]
after_s = ev[(ev['start_time'] >= ev["arrive"]) & 
            ((ev['dist_xita_m'] > R | ev['aoi'].isin(AOI2)))
            ]
n_ev = len(vis)
before, after = place_flags(before_s).sum(), place_flags(after_s).sum()
before['没有其他停留'] = n_ev - before_s['imsi'].nunique()
after['没有其他停留'] = n_ev - after_s['imsi'].nunique()

In [ ]:
# 离开商圈后多久到达西塔：每人在商圈最后一次停留的结束时候，到到达 之间隔了多久
gap = {}
for k in AOI2:
    last = before_s[before_s['aoi'] == k].groupby('imsi')['end_time'].max()
    gap[k] = ((arrive.reindex(last.index) - last).dt.total_seconds() /
                60).median()
link = pd.DataFrame({'之前人数': before, '之后人数': after}).astype(int)

link["之前占比"]  = (link["之前人数"] / n_ev).round(3)
link["之后占比"] = (link["之后人数"] /  n_ev).round(3)
link.rename_axis("地点").reset_index().to_sql(
    "ads_xita_linkage",con,if_exists="replace",index=False)
link

太原街：498 人来西塔之前在那里停留过（15%），离开后中位 70 分钟到西塔；之后再去太原街的只有 231 人。中街是 370 人，间隔 134 分钟，联动弱得多。

In [ ]:
# ── 出图 D8_03：对向条形图（一人可属多类，比例不能相加，所以不用桑基图）──
ROWS = ["没有其他停留", "其他地点", "过夜地", "中街商圈", "太原街商圈"]
fig, ax = plt.subplots(figsize=(10, 5.0), layout="constrained")
for i, k in enumerate(ROWS):
    hi = k in AOI2
    for val, sign, c_ in [(before[k] / n_ev, -1, MAIN if hi else GRAY), (after[k] / n_ev, 1, MAIN_MID if hi else "#DDE1E7")]:
        ax.barh(i, sign * val, color=c_, height=0.6)
        ax.text(sign * (val + 0.01), i, f"{val:.1%}", ha="right" if sign < 0 else "left", va="center", fontsize=9.5,
                color=INK, fontweight="bold" if hi else "normal")
ax.axvline(0, color=INK, lw=1)
ax.set_yticks(range(len(ROWS)), ROWS, fontsize=10, color=INK)
ax.set_xlim(-0.75, 0.75); ax.set_xticks([]); ax.grid(False); ax.set_ylim(-0.6, len(ROWS) + 0.1)
ax.text(-0.02, len(ROWS) - 0.35, "← 来西塔之前", ha="right", fontsize=10, color=MAIN, fontweight="bold")
ax.text(0.02, len(ROWS) - 0.35, "离开西塔之后 →", ha="left", fontsize=10, color=MAIN_DARK, fontweight="bold")
title = (f"{before['太原街商圈'] / n_ev:.0%} 的傍晚访客先在太原街停留再来西塔，"
         f"之后再去太原街的只有 {after['太原街商圈'] / n_ev:.0%}")
viz.finish(fig, title,
           f"{n_ev:,} 个傍晚访客当天其他停留的去处；一人可属多类，比例不能相加；“之前”= 傍晚到达西塔前结束，“之后”= 到达后开始；\n"
           "过夜地 = 离过夜地 500 米内；商圈 = v2 边界；口径：dws_stay、dws_night",
           SOURCE, "D8_03_商圈联动.png")
plt.show()

In [ ]:
findings = pd.DataFrame([
    ("F27", f"按过夜地划分，傍晚访客中市内来访占 {s_['市内来访'] / n_ev:.0%}，周边居民占 {s_['周边居民'] / n_ev:.0%}，{s_['过夜地不明'] / n_ev:.0%} 识别不出过夜地",
     "图 D8_01", "周边居民 = 过夜地距西塔 < 1.5 公里；过夜地不明不是一类人"),
    ("F28", f"仅按过夜地，外地来客只有 {old['外地来客']} 人（{old['外地来客'] / n_ev:.0%}）；加入当天入城轨迹后为 {new['外地来客']} 人（{new['外地来客'] / n_ev:.0%}），其中 {moved} 人原属“过夜地不明”",
     "图 D8_02", "外地来客 = 当天第一条信令在市域外、之后进入市域，或过夜地在市外；仍是下限"),
    ("F29", f"{before['太原街商圈'] / n_ev:.1%} 的傍晚访客到西塔前在太原街停留，离开后中位 {gap['太原街商圈']:.0f} 分钟到西塔；之后再去太原街的 {after['太原街商圈'] / n_ev:.1%}。中街为 {before['中街商圈'] / n_ev:.1%} 和 {after['中街商圈'] / n_ev:.1%}",
     "图 D8_03", "之前 = 傍晚到达西塔前结束的停留；之后 = 到达后开始的停留；一人可属多类；先后顺序不是因果"),
], columns=["编号", "发现", "证据", "口径"])
db.write_partition(con, findings.assign(来源="D08"), "ads_findings", "D08", dt_col="来源")
allf = pd.read_sql("SELECT * FROM ads_findings ORDER BY 编号", con)
allf[["编号", "发现", "来源"]]